# Graph Classification for PyTorch Geometric in TopologicPy

The goal is to classify the each graph based on its **graph label**. The graph label is categorical and is one of [0,1,2,3,4]

## What this notebook assumes

You already have a prepared dataset directory containing:

- `graphs.csv`
- `nodes.csv`
- `edges.csv`

## What this notebook does

1. Reads the dataset from disk
2. Loads the dataset into `topologicpy.PyG`
3. Set the model training hyperparameters
4. Trains a baseline **GraphSAGE** model for **graph classification**
5. Evaluates the model on validation and test graphs
6. Visualizes training history and confusion matrices


## Expected Folder Contents

- `graphs.csv`  
- `nodes.csv`  
- `edges.csv`  
- `meta.yaml`  

---

## This Script

1. Loads the dataset with the PyG helper class  
2. Builds a graph-level GNN model (graph classification by default)  
3. Trains with a train/val split  
4. Evaluates on validation and test sets  
5. Visualizes learning curves, confusion matrix, and prints evaluation metrics  

---

## Notes

- **Requires:** `topologicpy >= 0.9.31`, `torch`, `torch_geometric`, `pandas`, `pyyaml`, `numpy`, `plotly`, `scikit-learn`
- **Example Datasets can found at** `https://github.com/wassimj/topologicpy/tree/main/assets/MachineLearning`

### Installation Example

```bash
pip install torch pandas pyyaml numpy plotly scikit-learn
# then install torch-geometric following their official instructions for your OS/CUDA


In [41]:
# This cell is not needed if you have pip installed topologicpy
import sys
sys.path.append("C:/Users/sarwj/OneDrive - Cardiff University/Documents/GitHub/topologicpy/src")

## 1. Import the needed libraries and utility function

In [42]:
from __future__ import annotations
import os
import pandas as pd
from pathlib import Path
import json
import yaml
from topologicpy.PyG import PyG
from topologicpy.Helper import Helper

# def pretty_print_metrics(title: str, metrics: dict) -> None:
#     print("\n" + "=" * 80)
#     print(title)
#     print("=" * 80)
#     for k in sorted(metrics.keys()):
#         v = metrics[k]
#         if isinstance(v, float):
#             print(f"{k:30s}: {v:.6f}")
#         else:
#             print(f"{k:30s}: {v}")
#     print("=" * 80 + "\n")

def pretty_print_metrics(title: str, metrics: dict) -> None:
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)

    # Convert to DataFrame
    df = pd.DataFrame({
        "metric": list(metrics.keys()),
        "value": list(metrics.values())
    })

    # Sort by metric name (to preserve your original behaviour)
    df = df.sort_values("metric").reset_index(drop=True)

    # Format floats to 6 decimal places
    def _format(v):
        if isinstance(v, float):
            return f"{v:.6f}"
        return v

    df["value"] = df["value"].apply(_format)

    print(df.to_string(index=False))

    print("=" * 80 + "\n")

## 2. Check the TopologicPy Version

In [43]:
print("The script is compatible with TopologicPy v0.9.31 or newer.")
print(Helper.Version())

The script is compatible with TopologicPy v0.9.31 or newer.
The version that you are using (0.9.18) is OLDER than the latest version (0.9.43) from PyPI. Please consider upgrading to the latest version.


## 3. Set your renderer:
* Visual studio code: "vscode"
* Google Colab: "colab"
* Browser: "browser"

In [45]:
renderer = "vscode"

## Baseline
```
# ------------------------------------------------------------------
# DATASET LOCATION
# ------------------------------------------------------------------
# Change this path if your prepared clean dataset is elsewhere.
# The folder must contain graphs.csv, nodes.csv, and edges.csv.
DATASET_PATH = Path(r"C:\Users\sarwj\OneDrive - Cardiff University\IAAC\2025-26\S3 - Buildings As Graphs\notebooks\Supporting Files\dataset_graph_classification").resolve()
MODEL_PATH = Path(r"C:\Users\sarwj\OneDrive - Cardiff University\Desktop\pyg_model.pt").resolve()

# ------------------------------------------------------------------
# TASK DEFINITION
# ------------------------------------------------------------------
PREDICTION_LEVEL = "graph"              # Prediction target level: graph | node | edge | link.
TASK = "classification"                 # Learning task type: classification | regression | link_prediction.
GRAPH_LABEL_TYPE = "categorical"        # Type of graph-level label: continuous for regression targets such as cooling/heating load.
NODE_LABEL_TYPE = "categorical"         # Type of node-level label: categorical class index, e.g. room type or element type.
EDGE_LABEL_TYPE = "categorical"         # Type of edge-level label: categorical class index; currently not used in this setup.
HOLDOUT_GROUP_BY = "label"              # Grouping key for holdout splitting; keeps items with the same label together where supported.


# ------------------------------------------------------------------
# MODEL DEFINITION
# ------------------------------------------------------------------
CONV = "sage"                           # Graph convolution operator to use; "sage" selects GraphSAGE.
HIDDEN_DIMS = (128, 128)                # Hidden layer dimensions; two values define two hidden graph-convolution layers.
ACTIVATION = "relu"                     # Activation function used between hidden layers.
DROPOUT = 0.20                          # Dropout probability applied during training to reduce overfitting.
BATCH_NORM = True                       # Whether to apply batch normalisation after hidden layers.
RESIDUAL = True                         # Whether to use residual/skip connections between compatible layers.
POOLING = "mean"                        # Graph-level pooling operation used to aggregate node embeddings into graph embeddings.


# ------------------------------------------------------------------
# DATA SPLIT SETTINGS
# ------------------------------------------------------------------
CROSS_VALIDATION = "holdout"            # Validation strategy; "holdout" uses one train/validation/test split.
TRAIN_RATIO = 0.8                       # Proportion of data assigned to the training set.
VAL_RATIO = 0.1                         # Proportion of data assigned to the validation set.
TEST_RATIO = 0.1                        # Proportion of data assigned to the test set.
RANDOM_STATE = 42                       # Random seed used for reproducible splitting, shuffling, and training initialisation where supported.
SHUFFLE = True                          # Whether to shuffle the dataset before splitting.


# ------------------------------------------------------------------
# TRAINING SETTINGS
# ------------------------------------------------------------------
LEARNING_RATE = 1e-3                    # Optimiser learning rate.
BATCH_SIZE = 64                         # Number of graphs/samples per training batch.
EPOCHS = 50                             # Maximum number of training epochs.
OPTIMIZER = "adamw"                     # Optimiser to use for training; AdamW includes decoupled weight decay.
WEIGHT_DECAY = 1e-4                     # L2 regularisation strength used by the optimiser.
GRADIENT_CLIP_NORM = 1.0                # Maximum gradient norm for clipping; helps stabilise training.
EARLY_STOPPING = True                   # Whether to stop training early when validation performance stops improving.
EARLY_STOPPING_PATIENCE = 12            # Number of epochs without validation improvement before early stopping is triggered.
USE_GPU = True                          # Whether to use a CUDA-capable GPU when available.

```

## 4. Specify input parameters

In [46]:
# ------------------------------------------------------------------
# DATASET LOCATION
# ------------------------------------------------------------------
# Change this path if your prepared clean dataset is elsewhere.
# The folder must contain graphs.csv, nodes.csv, and edges.csv.
# Training dataset (1495 graphs, 5 classes) downloaded from topologicpy assets/MachineLearning/training_dataset.
DATASET_PATH = Path(r"D:\IAAC\A3_GraphML_GraphClassification\training_dataset").resolve()
MODEL_PATH = Path(r"D:\IAAC\A3_GraphML_GraphClassification\ModelPath\pyg_model.pt").resolve()  # local: Phase 1 saves here, Phase 2 loads here

# ------------------------------------------------------------------
# TASK DEFINITION
# ------------------------------------------------------------------
PREDICTION_LEVEL = "graph"              # Prediction target level: graph | node | edge | link.
TASK = "classification"                 # Learning task type: classification | regression | link_prediction.
GRAPH_LABEL_TYPE = "categorical"        # Type of graph-level label: continuous for regression targets such as cooling/heating load.
NODE_LABEL_TYPE = "categorical"         # Type of node-level label: categorical class index, e.g. room type or element type.
EDGE_LABEL_TYPE = "categorical"         # Type of edge-level label: categorical class index; currently not used in this setup.
HOLDOUT_GROUP_BY = "label"              # Grouping key for holdout splitting; keeps items with the same label together where supported.


# ------------------------------------------------------------------
# MODEL DEFINITION
# ------------------------------------------------------------------
CONV = "sage"                           # Graph convolution operator to use; "sage" selects GraphSAGE.
HIDDEN_DIMS = (128, 128)                # Hidden layer dimensions; two values define two hidden graph-convolution layers.
ACTIVATION = "relu"                     # Activation function used between hidden layers.
DROPOUT = 0.20                          # Dropout probability applied during training to reduce overfitting.
BATCH_NORM = True                       # Whether to apply batch normalisation after hidden layers.
RESIDUAL = True                         # Whether to use residual/skip connections between compatible layers.
POOLING = "mean"                        # Graph-level pooling operation used to aggregate node embeddings into graph embeddings.


# ------------------------------------------------------------------
# DATA SPLIT SETTINGS
# ------------------------------------------------------------------
CROSS_VALIDATION = "holdout"            # Validation strategy; "holdout" uses one train/validation/test split.
TRAIN_RATIO = 0.8                       # Proportion of data assigned to the training set.
VAL_RATIO = 0.1                         # Proportion of data assigned to the validation set.
TEST_RATIO = 0.1                        # Proportion of data assigned to the test set.
RANDOM_STATE = 42                       # Random seed used for reproducible splitting, shuffling, and training initialisation where supported.
SHUFFLE = True                          # Whether to shuffle the dataset before splitting.


# ------------------------------------------------------------------
# TRAINING SETTINGS
# ------------------------------------------------------------------
LEARNING_RATE = 1e-3                    # Optimiser learning rate.
BATCH_SIZE = 64                         # Number of graphs/samples per training batch.
EPOCHS = 50                             # Maximum number of training epochs.
OPTIMIZER = "adamw"                     # Optimiser to use for training; AdamW includes decoupled weight decay.
WEIGHT_DECAY = 1e-4                     # L2 regularisation strength used by the optimiser.
GRADIENT_CLIP_NORM = 1.0                # Maximum gradient norm for clipping; helps stabilise training.
EARLY_STOPPING = True                   # Whether to stop training early when validation performance stops improving.
EARLY_STOPPING_PATIENCE = 12            # Number of epochs without validation improvement before early stopping is triggered.
USE_GPU = True                          # Whether to use a CUDA-capable GPU when available.


## 5. Load the CSV Dataset (The Example Has Categorical Labels, Task is Graph-level Classification)

In [47]:
# Optional: read meta.yaml (purely informational)
meta_path = DATASET_PATH / "meta.yaml"
if meta_path.exists():
    meta = yaml.safe_load(meta_path.read_text(encoding="utf-8"))
    print("Loaded meta.yaml:")
    print(json.dumps(meta, indent=2))
else:
    print("meta.yaml not found (this is fine).")

# This dataset has graphs.csv with a categorical 'label' column -> graph classification.
pyg = PyG.ByCSVPath(
    path=str(DATASET_PATH),
    level=PREDICTION_LEVEL,
    task=TASK,
    graphLabelType=GRAPH_LABEL_TYPE,
    nodeLabelType=NODE_LABEL_TYPE,
    edgeLabelType=EDGE_LABEL_TYPE,

    # If your headers differ, override here (your attached CSVs match defaults):
    # graphIDHeader="graph_id", graphLabelHeader="label",
    # nodeIDHeader="node_id", nodeLabelHeader="label",
    # edgeSRCHeader="src_id", edgeDSTHeader="dst_id", edgeLabelHeader="label",
)

Loaded meta.yaml:
{
  "dataset_name": "topologic_training_dataset",
  "edge_data": [
    {
      "file_name": "edges.csv"
    }
  ],
  "node_data": [
    {
      "file_name": "nodes.csv"
    }
  ],
  "graph_data": {
    "file_name": "graphs.csv"
  }
}


## 6. Set Hyperparameters

In [48]:
pyg.SetHyperparameters(
    # splitting / determinism
    cv=CROSS_VALIDATION,
    split=(TRAIN_RATIO, VAL_RATIO, TEST_RATIO),
    random_state=RANDOM_STATE,
    shuffle=SHUFFLE,

    # training
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    optimizer=OPTIMIZER,
    gradient_clip_norm=GRADIENT_CLIP_NORM,
    early_stopping=EARLY_STOPPING,
    early_stopping_patience=EARLY_STOPPING_PATIENCE,
    use_gpu=USE_GPU,

    # model
    conv=CONV,
    hidden_dims=HIDDEN_DIMS,
    activation=ACTIVATION,
    dropout=DROPOUT,
    batch_norm=BATCH_NORM,
    residual=RESIDUAL,
    pooling=POOLING
)
# Print a compact summary of the current config and inferred dims/classes
print("PyG config summary:")
print(pyg.Summary())

PyG config summary:
{'level': 'graph', 'task': 'classification', 'graph_label_type': 'categorical', 'node_label_type': 'categorical', 'edge_label_type': 'categorical', 'cv': 'holdout', 'split': (0.8, 0.1, 0.1), 'k_folds': 5, 'conv': 'sage', 'hidden_dims': (128, 128), 'activation': 'relu', 'dropout': 0.2, 'batch_norm': True, 'residual': True, 'pooling': 'mean', 'epochs': 50, 'batch_size': 64, 'lr': 0.001, 'weight_decay': 0.0001, 'optimizer': 'adamw', 'gradient_clip_norm': 1.0, 'early_stopping': True, 'early_stopping_patience': 12, 'device': 'cpu', 'num_graphs': 1496, 'num_outputs': 5}


## 6. Train the Model

In [49]:
history = pyg.Train()  # returns dict of per-epoch curves (loss + metrics when available)
print(history)

{'train_loss': [0.979257975753985, 0.46531777632863897, 0.28005645541768326, 0.21763934430323148, 0.16938206906381406, 0.15163792983481758, 0.1424206207065206, 0.12703983211203626, 0.11746600976115779, 0.10753907822072506, 0.10003619435194291, 0.07737344101463493, 0.0795202210153404, 0.07109318506952964, 0.062346156196374646, 0.06074108337787421, 0.06796039523262727, 0.052338633588270136, 0.05235994724850906, 0.05096084776481515, 0.05091442814783046, 0.0490740197465608, 0.0421421954310254, 0.03434199364365716, 0.0384992682129929, 0.03742975837207938, 0.05998367375056995, 0.0357888956603251, 0.035062828517862056, 0.036717728098952455, 0.030782112508620087, 0.031162805389612913, 0.04090657756712876, 0.05148298327664012, 0.030161871366496933, 0.030466533731669188, 0.030072633642703295, 0.02609167700192254, 0.02497877262679762, 0.03891609482908327, 0.027740882734130872, 0.02316196042260057, 0.025797119878820683, 0.024661370430533822, 0.02386797579789632, 0.016760715283453465, 0.02250562712

## 7. Plot the Training and Validation Loss Curves

In [50]:
fig_hist = pyg.PlotHistory()
fig_hist.show()

## 8. Validate the Model

In [51]:
val_metrics = pyg.Validate()
pretty_print_metrics("Validation metrics", val_metrics)



Validation metrics
       metric    value
 val_accuracy 1.000000
       val_f1 1.000000
val_precision 1.000000
   val_recall 1.000000



## 9. Test the Model

In [54]:
test_metrics = pyg.Test()
pretty_print_metrics("Test metrics", test_metrics)


Test metrics
        metric    value
 test_accuracy 1.000000
       test_f1 1.000000
test_precision 1.000000
   test_recall 1.000000



## 10. Plot the Confusion Matrix (For Categorical Labels)

### For training portion

In [55]:
fig_cm = pyg.PlotConfusionMatrix(split="train")
fig_cm.show()

### For validation portion

In [56]:
fig_cm = pyg.PlotConfusionMatrix(split="validate")
fig_cm.show()

### For testing portion

In [57]:
fig_cm = pyg.PlotConfusionMatrix(split="test")
fig_cm.show()

### For the whole dataset

In [58]:
fig_cm = pyg.PlotConfusionMatrix(split="all")
fig_cm.show()

## 11. Save the Model

In [59]:
pyg.SaveModel(str(MODEL_PATH))

# ----- PHASE 2: PREDICTION OF UNSEEN DATASET ------

## 1. Load Testing Dataset

In [60]:
#dataset_dir = Path(r"C:\Users\sarwj\OneDrive - Cardiff University\Documents\GitHub\topologicpy\assets\MachineLearning\testing_dataset").resolve()

# Your exported building graph (from the S06-13A notebook).
dataset_dir = Path(r"D:\IAAC\A3_GraphML_GraphClassification\brgdataset").resolve()

mapping = {0: "Separation",
           1: "Separation with Plinth",
           2: "Adherence",
           3: "Adherence with Plinth",
           4: "Interlock"}

pyg_2 = PyG.ByCSVPath(
    path=str(dataset_dir),
    level=PREDICTION_LEVEL,
    task=TASK,
    graphLabelType=GRAPH_LABEL_TYPE,
    nodeLabelType=NODE_LABEL_TYPE,
    edgeLabelType=EDGE_LABEL_TYPE,
    # If your headers differ, override here (your attached CSVs match defaults):
    # graphIDHeader="graph_id", graphLabelHeader="label",
    # nodeIDHeader="node_id", nodeLabelHeader="label",
    # edgeSRCHeader="src_id", edgeDSTHeader="dst_id", edgeLabelHeader="label",
)

## 2. Load the Pre-trained Model

In [61]:
pyg_2.LoadModel(str(MODEL_PATH))

## 3. Make the Whole Dataset a Testing Dataset

In [62]:
pyg_2.SetHyperparameters(split=(0.0, 0.0, 1.0), shuffle=False)  # all graphs become test
print("PyG config summary:")
print(pyg_2.Summary())

PyG config summary:
{'level': 'graph', 'task': 'classification', 'graph_label_type': 'categorical', 'node_label_type': 'categorical', 'edge_label_type': 'categorical', 'cv': 'holdout', 'split': (0.0, 0.0, 1.0), 'k_folds': 5, 'conv': 'sage', 'hidden_dims': (128, 128), 'activation': 'relu', 'dropout': 0.2, 'batch_norm': True, 'residual': True, 'pooling': 'mean', 'epochs': 50, 'batch_size': 32, 'lr': 0.001, 'weight_decay': 0.0, 'optimizer': 'adam', 'gradient_clip_norm': None, 'early_stopping': False, 'early_stopping_patience': 10, 'device': 'cpu', 'num_graphs': 1, 'num_outputs': 5}


## 4. Predict the Dataset

In [63]:
pred_results = pyg_2.Predict()
indices = pred_results['index'].tolist()
predictions = pred_results['pred'].tolist()
probabilities = pred_results['prob'].tolist()


prediction_labels = [mapping[prediction] for prediction in predictions]
df = pd.DataFrame({
    "index": indices,
    "prediction": predictions,
    "Label": prediction_labels,
    "confidence": [round(max(p), 2) for p in probabilities]
})

df

,index,prediction,Label,confidence
0,0,0,Separation,1.0


## 5. Plot the Confusion Matrix (For Categorical Labels Only)

In [64]:
import numpy as np
from sklearn.metrics import confusion_matrix
from topologicpy.Plotly import Plotly
actual = np.array(pred_results["y_true"]).reshape(-1)
predicted = np.array(pred_results["pred"]).reshape(-1)
cm = confusion_matrix(actual, predicted, labels=[0,1,2,3,4])
fig = Plotly.FigureByConfusionMatrix(cm)
Plotly.Show(fig)